# GUS04D — Variable-Specific Estimation Pipelines

Run demographic estimation for all variable types using the three-layer pipeline:
- **Layer 1**: Log-linear interpolation (seed generation)
- **Layer 2**: Marginal fitting via IPF (not used in age×sex_2000)
- **Layer 3**: Hierarchical consistency (residual scaling / Gurobi QP)

**Prerequisites**: Run GUS02B → GUS03 to produce `geoteryt_O.pkl`.

In [1]:
# ── Cell 1: Imports & load database ──
import sys, os, time
import numpy as np
import pandas as pd

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'Code', 'tools'))
DATA_ROOT = os.path.join(REPO, '..', '..', 'Data', 'Geospatial')

from geoTERYT_db import (
    load_complete_database, LEVEL_GMINA, LEVEL_VOIVODESHIP, LEVEL_POWIAT,
)

db_path = os.path.join(DATA_ROOT, 'geoteryt_O.pkl')
print(f"Loading database from {db_path}…")
t0 = time.time()
db = load_complete_database(db_path)
print(f"Loaded in {time.time()-t0:.1f}s — {len(db._records)} records")

Loading database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_O.pkl…
Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_O.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4612 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3661
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ Records with population data: 4582
  ✓ Records with pop_class: 3411
Loaded in 348.6s — 4612 records


In [7]:
# ── Cell 2: Initialize DemographicEstimator ──
import importlib
import demographic_estimator
importlib.reload(demographic_estimator)
from demographic_estimator import DemographicEstimator, PREDICTION_2000_RANGE, PREDICTION_1990_RANGE

est = DemographicEstimator(db, verbose=True)
print(repr(est))

DemographicEstimator initialised  (Gurobi=YES, IPFN=YES)
DemographicEstimator(completed=0/9, Gurobi=YES)


## Age × Sex — Prediction2000 (1999–2025)

Sources: M_pop__age_sex (BDL P2137 + Census 2002/2011/2021), shape (19, 3).  
Coverage: ~87–95% of gminas per year. Missing gminas filled via log-linear interpolation + voivodeship residual scaling.

In [8]:
# ── Cell 3: Run age×sex Prediction2000 ──
t0 = time.time()
est.run_pipeline('age_sex', '2000')
print(f"\nCompleted in {time.time()-t0:.1f}s")


  PIPELINE: age_sex / Prediction2000
  Output subject: E_age_sex_2000
  Source: M_age_sex  shape=(16, 3)
  Gminas total: 2671, with M_age_sex: 2671
  Layer 1: generating seeds (log-linear interpolation)…
    Seeds generated: 2671/2671 units (skipped 0)
    2000: 2490 obs + 0 est
    2005: 2479 obs + 0 est
    2010: 2480 obs + 0 est
    2015: 2479 obs + 0 est
    2020: 2478 obs + 0 est
    2025: 0 obs + 2478 est
  Aggregating to powiat and voivodeship levels…
    Aggregated: 10229 powiat-years, 432 voiv-years
  Summary: 64486 observed + 2478 estimated cell-years stored
  ✓  E_age_sex_2000 complete

Completed in 15.1s


In [9]:
# ── Cell 4: Validation — coverage and provenance ──
e_sid = 'E_age_sex_2000'

# Count records with E_ data by level
by_level = {'gmina': 0, 'powiat': 0, 'voiv': 0}
by_level_total = {'gmina': 0, 'powiat': 0, 'voiv': 0}

for tid, rec in db._records.items():
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}:
        by_level_total['gmina'] += 1
        if e_sid in rec.cross_tables:
            ct = rec.cross_tables[e_sid]
            if ct.years_with_data:
                by_level['gmina'] += 1
    elif rec.level == LEVEL_POWIAT:
        by_level_total['powiat'] += 1
        if e_sid in rec.cross_tables:
            ct = rec.cross_tables[e_sid]
            if ct.years_with_data:
                by_level['powiat'] += 1
    elif rec.level == LEVEL_VOIVODESHIP:
        by_level_total['voiv'] += 1
        if e_sid in rec.cross_tables:
            ct = rec.cross_tables[e_sid]
            if ct.years_with_data:
                by_level['voiv'] += 1

print(f"E_age_sex_2000 coverage:")
for lvl in ['gmina', 'powiat', 'voiv']:
    print(f"  {lvl}: {by_level[lvl]}/{by_level_total[lvl]}")

# Provenance summary
prov_df = est.get_provenance_summary(e_sid)
if not prov_df.empty:
    print(f"\nProvenance (by year):")
    print(prov_df.to_string())

E_age_sex_2000 coverage:
  gmina: 2671/2671
  powiat: 382/382
  voiv: 16/67

Provenance (by year):
      n_units  mean_frac_observed  min_frac_observed
year                                                
1999     3069            0.811339                0.0
2000     3069            0.811339                0.0
2001     3069            0.811339                0.0
2002     3069            0.807755                0.0
2003     3069            0.807755                0.0
2004     3069            0.807755                0.0
2005     3069            0.807755                0.0
2006     3069            0.807755                0.0
2007     3069            0.807755                0.0
2008     3069            0.807755                0.0
2009     3069            0.807755                0.0
2010     3069            0.808081                0.0
2011     3069            0.808081                0.0
2012     3069            0.808081                0.0
2013     3069            0.808081                0.0


In [10]:
# ── Cell 5: Validation — voivodeship consistency ──
# Compare estimated voivodeship E_ totals with observed M_pop__age_sex

source_sid = 'M_pop__age_sex'
e_sid = 'E_age_sex_2000'
voiv_tids = sorted(tid for tid in db._by_level.get(LEVEL_VOIVODESHIP, set()))

def _ogolem_grand(ct, tbl):
    """Get ogółem×ogółem cell value from a 2D cross table."""
    d = ct.dim_names
    l = ct.dim_labels
    og0 = next((i for i, lb in enumerate(l[d[0]]) if lb.lower()=='ogółem'), None)
    og1 = next((i for i, lb in enumerate(l[d[1]]) if lb.lower()=='ogółem'), None)
    if og0 is not None and og1 is not None:
        return tbl[og0, og1]
    return np.nansum(tbl)

errors = []
for voiv_tid in voiv_tids:
    rec = db._records.get(voiv_tid)
    if rec is None:
        continue
    obs_ct = rec.cross_tables.get(source_sid)
    est_ct = rec.cross_tables.get(e_sid)
    if obs_ct is None or est_ct is None:
        continue
    
    for year in PREDICTION_2000_RANGE:
        obs_tbl = obs_ct.tables.get(year)
        est_tbl = est_ct.tables.get(year)
        if obs_tbl is None or np.all(np.isnan(obs_tbl)):
            continue
        if est_tbl is None or np.all(np.isnan(est_tbl)):
            continue
        
        obs_grand = _ogolem_grand(obs_ct, obs_tbl)
        est_grand = _ogolem_grand(est_ct, est_tbl)
        
        if obs_grand > 0:
            pct_err = 100 * (est_grand - obs_grand) / obs_grand
            errors.append({
                'voiv': voiv_tid, 'name': rec.name, 'year': year,
                'obs_pop': obs_grand, 'est_pop': est_grand,
                'pct_error': pct_err
            })

if errors:
    err_df = pd.DataFrame(errors)
    print(f"Voivodeship consistency: {len(err_df)} comparisons")
    print(f"  Mean abs % error: {err_df['pct_error'].abs().mean():.3f}%")
    print(f"  Max  abs % error: {err_df['pct_error'].abs().max():.3f}%")
    worst = err_df.loc[err_df['pct_error'].abs().idxmax()]
    print(f"  Worst: {worst['name']} ({worst['voiv']}) year {worst['year']}: "
          f"{worst['pct_error']:.3f}%")
else:
    print("No voivodeship comparisons available")

Voivodeship consistency: 416 comparisons
  Mean abs % error: 2.058%
  Max  abs % error: 33.837%
  Worst: MAZOWIECKIE (1400000) year 2024: 33.837%


In [11]:
# ── Cell 6: Spot-check — sample gmina time series ──
# Pick 3 gminas: one with full data, one with partial data, one estimated-heavy

sample_tids = []
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect = rec.cross_tables.get(e_sid)
    if ect is None:
        continue
    ywd = ect.years_with_data
    if len(sample_tids) == 0 and len(ywd) >= 25:
        sample_tids.append(tid)  # well-covered
    elif len(sample_tids) == 1 and 10 <= len(ywd) < 25:
        sample_tids.append(tid)  # partial
    elif len(sample_tids) == 2:
        break
# If we didn't find a partial one, just use a different well-covered one
if len(sample_tids) < 2:
    for tid, rec in db._records.items():
        if tid not in sample_tids and rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}:
            ect = rec.cross_tables.get(e_sid)
            if ect and ect.years_with_data:
                sample_tids.append(tid)
                break

for tid in sample_tids[:3]:
    rec = db._records[tid]
    ect = rec.cross_tables[e_sid]
    print(f"\n{'='*60}")
    print(f"{rec.name} ({tid}) — {len(ect.years_with_data)} years with E_ data")
    
    # Grand total from E_ table (use its own dim_labels for ogółem)
    e_og0 = next((i for i, l in enumerate(ect.dim_labels[ect.dim_names[0]]) if l.lower()=='ogółem'), None)
    e_og1 = next((i for i, l in enumerate(ect.dim_labels[ect.dim_names[1]]) if l.lower()=='ogółem'), None) if len(ect.dim_names) > 1 else None
    
    # Grand total from source M_ table (use ITS own dim_labels)
    source_ct = rec.cross_tables.get(source_sid)
    m_og0, m_og1 = None, None
    if source_ct:
        m_og0 = next((i for i, l in enumerate(source_ct.dim_labels[source_ct.dim_names[0]]) if l.lower()=='ogółem'), None)
        m_og1 = next((i for i, l in enumerate(source_ct.dim_labels[source_ct.dim_names[1]]) if l.lower()=='ogółem'), None) if len(source_ct.dim_names) > 1 else None
    
    print(f"Year | E_total  | pop      | source_M | match?")
    print(f"-----|----------|----------|----------|-------")
    for year in PREDICTION_2000_RANGE:
        tbl = ect.tables.get(year)
        if tbl is None or np.all(np.isnan(tbl)):
            continue
        e_total = tbl[e_og0, e_og1] if (e_og0 is not None and e_og1 is not None) else (tbl[e_og0] if e_og0 is not None else np.nansum(tbl))
        
        pop_val = rec.pop.get(pd.Timestamp(year, 1, 1), np.nan)
        
        m_total = np.nan
        if source_ct:
            m_tbl = source_ct.tables.get(year)
            if m_tbl is not None and not np.all(np.isnan(m_tbl)):
                if m_og0 is not None and m_og1 is not None:
                    m_total = m_tbl[m_og0, m_og1]
                elif m_og0 is not None:
                    m_total = m_tbl[m_og0]
                else:
                    m_total = np.nansum(m_tbl)
        
        is_obs = "OBS" if not np.isnan(m_total) else "EST"
        match = "✓" if abs(e_total - pop_val) < 1 else f"Δ={e_total-pop_val:.1f}" if not np.isnan(pop_val) else "?"
        print(f"{year} | {e_total:>8.0f} | {pop_val:>8.0f} | {m_total:>8.0f} | {is_obs} {match}")


Bolesławiec (0201011) — 27 years with E_ data
Year | E_total  | pop      | source_M | match?
-----|----------|----------|----------|-------
1999 |    41914 |    41914 |    41914 | OBS ✓
2000 |    41731 |    41731 |    41731 | OBS ✓
2001 |    41646 |    41646 |    41646 | OBS ✓
2002 |    41371 |    41371 |    41371 | OBS ✓
2003 |    41263 |    41263 |    41263 | OBS ✓
2004 |    41117 |    41117 |    41117 | OBS ✓
2005 |    40984 |    40984 |    40984 | OBS ✓
2006 |    40679 |    40679 |    40679 | OBS ✓
2007 |    40384 |    40384 |    40384 | OBS ✓
2008 |    40258 |    40258 |    40258 | OBS ✓
2009 |    40021 |    40021 |    40021 | OBS ✓
2010 |    40309 |    40309 |    40309 | OBS ✓
2011 |    40119 |    40119 |    40119 | OBS ✓
2012 |    39851 |    39851 |    39851 | OBS ✓
2013 |    39603 |    39603 |    39603 | OBS ✓
2014 |    39464 |    39464 |    39464 | OBS ✓
2015 |    39373 |    39373 |    39373 | OBS ✓
2016 |    39167 |    39167 |    39167 | OBS ✓
2017 |    39084 |    39084 |   

## Age × Sex — Prediction1990 (1986–2002)

Sources: M_age_sex (16×3): P2137 (BDL gmina 1995–2024) + H_age_sex (old voivodeships 1986–1994).  
Census 1988: P2884 (age 10yr bins, gmina) + P2883 (sex, gmina).  
Challenge: 1988 census gives age/sex separately → must construct 2D joint table via IPF.

In [12]:
# ── Cell 7: Run age×sex Prediction1990 ──
import importlib, demographic_estimator
importlib.reload(demographic_estimator)
from demographic_estimator import DemographicEstimator

est = DemographicEstimator(db, verbose=True)
est._completed.add(('age_sex', '2000'))

t0 = time.time()
est.run_pipeline('age_sex', '1990')
print(f"\nCompleted in {time.time()-t0:.1f}s")

DemographicEstimator initialised  (Gurobi=YES, IPFN=YES)

  PIPELINE: age_sex / Prediction1990
  Output subject: E_age_sex_1990
  Source: M_age_sex  shape=(16, 3)
  Phase A: constructing 1988 gmina age×sex via IPF…
    1988 IPF: 2478 OK, 193 skipped
  Phase B: building seeds (log-linear interpolation)…
    Seeds: 2671 gminas
  Phase C: old voivodeship marginal scaling (1986–1994)…
    Scaled 441 old-voi × year combinations
  Phase D: storing results…
  Summary: 45407 cell-years for 2671 gminas
  ✓  E_age_sex_1990 complete

Completed in 21.9s


In [13]:
# ── Cell 8: Validation — E_age_sex_1990 quality checks ──
e_sid_1990 = 'E_age_sex_1990'
source_sid_1990 = 'M_age_sex'

# 1. Coverage
n_gminas_with = sum(
    1 for tid, rec in db._records.items()
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
    and e_sid_1990 in rec.cross_tables
    and rec.cross_tables[e_sid_1990].years_with_data
)
print(f"E_age_sex_1990 coverage: {n_gminas_with} gminas")

# 2. Census 1988 vs population consistency
errors_1988 = []
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect = rec.cross_tables.get(e_sid_1990)
    if ect is None:
        continue
    tbl = ect.tables.get(1988)
    if tbl is None or np.all(np.isnan(tbl)):
        continue
    
    # Find ogółem×ogółem (grand total)
    og0 = next((i for i, l in enumerate(ect.dim_labels[ect.dim_names[0]]) if l.lower()=='ogółem'), None)
    og1 = next((i for i, l in enumerate(ect.dim_labels[ect.dim_names[1]]) if l.lower()=='ogółem'), None)
    if og0 is None or og1 is None:
        continue
    e_total = tbl[og0, og1]
    pop88 = rec.pop.get(pd.Timestamp(1988,1,1), np.nan)
    if not np.isnan(pop88) and pop88 > 0:
        pct_err = 100 * (e_total - pop88) / pop88
        errors_1988.append({'tid': tid, 'name': rec.name, 'e_total': e_total, 'pop': pop88, 'pct_err': pct_err})

err_df = pd.DataFrame(errors_1988)
if not err_df.empty:
    print(f"\n1988 E_ vs pop consistency ({len(err_df)} gminas):")
    print(f"  Mean abs % error: {err_df['pct_err'].abs().mean():.4f}%")
    print(f"  Max  abs % error: {err_df['pct_err'].abs().max():.4f}%")
    print(f"  Within 1%: {(err_df['pct_err'].abs() < 1).sum()}/{len(err_df)}")
    print(f"  Within 5%: {(err_df['pct_err'].abs() < 5).sum()}/{len(err_df)}")

# 3. Old voivodeship consistency for 1988
print(f"\n1988 old-voivodeship consistency:")
country_rec = db._records.get('0000000')
old_voi_tids = country_rec.children_ids.get('old', []) if country_rec else []
ov_errors = []
for ov_tid in old_voi_tids:
    ov_rec = db._records.get(ov_tid)
    if ov_rec is None:
        continue
    ov_ct = ov_rec.cross_tables.get(source_sid_1990)
    if ov_ct is None:
        continue
    ov_tbl = ov_ct.tables.get(1988)
    if ov_tbl is None or np.all(np.isnan(ov_tbl)):
        continue
    
    # Sum gmina E_ tables
    children = ov_rec.get_children(1988)
    agg = np.zeros_like(ov_tbl)
    n_children = 0
    for g in children:
        grec = db._records.get(g)
        if grec is None:
            continue
        gct = grec.cross_tables.get(e_sid_1990)
        if gct is None:
            continue
        gtbl = gct.tables.get(1988)
        if gtbl is None or np.all(np.isnan(gtbl)):
            continue
        if gtbl.shape != ov_tbl.shape:
            continue
        agg += np.nan_to_num(gtbl, nan=0.0)
        n_children += 1
    
    og0 = next((i for i, l in enumerate(ov_ct.dim_labels[ov_ct.dim_names[0]]) if l.lower()=='ogółem'), None)
    og1 = next((i for i, l in enumerate(ov_ct.dim_labels[ov_ct.dim_names[1]]) if l.lower()=='ogółem'), None)
    if og0 is not None and og1 is not None:
        ov_total = ov_tbl[og0, og1]
        agg_total = agg[og0, og1]
        if ov_total > 0:
            pct = 100 * (agg_total - ov_total) / ov_total
            ov_errors.append({'ov': ov_tid, 'name': ov_rec.name, 'pct_err': pct, 'n_children': n_children})

ov_df = pd.DataFrame(ov_errors)
if not ov_df.empty:
    print(f"  {len(ov_df)} old voivodeships compared")
    print(f"  Mean abs % error: {ov_df['pct_err'].abs().mean():.4f}%")
    print(f"  Max  abs % error: {ov_df['pct_err'].abs().max():.4f}%")
    worst = ov_df.loc[ov_df['pct_err'].abs().idxmax()]
    print(f"  Worst: {worst['name']} ({worst['ov']}): {worst['pct_err']:.2f}% ({int(worst['n_children'])} children)")

E_age_sex_1990 coverage: 2671 gminas

1988 E_ vs pop consistency (2490 gminas):
  Mean abs % error: 0.0000%
  Max  abs % error: 0.0000%
  Within 1%: 2490/2490
  Within 5%: 2490/2490

1988 old-voivodeship consistency:
  49 old voivodeships compared
  Mean abs % error: 4.6101%
  Max  abs % error: 28.1156%
  Worst: Wałbrzyskie (5700000): 28.12% (49 children)


In [14]:
# Very concise Warsaw check
p = db.get_by_teryt_id('1431000')
for yr in [1999, 2000, 2001, 2002, 2003]:
    ch = p.get_children(yr) if p else []
    r123 = [g for g in ch if db.get_by_teryt_id(g) and db.get_by_teryt_id(g).rodz in ('1','2','3')]
    t = sum(db.get_by_teryt_id(g).pop.get(pd.Timestamp(yr,1,1), 0) for g in r123)
    pp = p.pop.get(pd.Timestamp(yr,1,1), np.nan) if p else np.nan
    ww = [g for g in r123 if g.startswith('1431')]
    print(f"yr={yr}: n_r123={len(r123):3d}  n_warsaw={len(ww):2d}  sum_pop={t:.0f}  pow_pop={pp:.0f}  ratio={t/pp:.3f}" if pp else f"yr={yr}: no pop")

yr=1999: n_r123= 12  n_warsaw=12  sum_pop=3354632  pow_pop=1677316  ratio=2.000
yr=2000: n_r123= 12  n_warsaw=12  sum_pop=3344836  pow_pop=1672418  ratio=2.000
yr=2001: n_r123= 12  n_warsaw=12  sum_pop=3343454  pow_pop=1671727  ratio=2.000
yr=2002: n_r123= 12  n_warsaw=12  sum_pop=nan  pow_pop=1688194  ratio=nan
yr=2003: n_r123= 12  n_warsaw=12  sum_pop=nan  pow_pop=1689559  ratio=nan


## Education — Prediction2000 (1999–2025)

Source: M_educ_2000 (1D, 5 categories, **no ogółem**): P2402 + P3309 + P4315 + P2350 + P4092.  
Strategy: 2011 powiat census disaggregated to gmina → 3-anchor spline interpolation → Layer 2 voivodeship marginal scaling.  
Consistency check: voivodeship-level M_educ_2000 sums vs aggregated E_ gmina sums.

In [15]:
# ── Run Education Prediction2000 ──
t0 = time.time()
est.run_pipeline('educ', '2000')
print(f"\nCompleted in {time.time()-t0:.1f}s")


  PIPELINE: educ / Prediction2000
  Output subject: E_educ_2000
  Source: M_educ_2000  shape=(5,)
  Disaggregating 2011 powiat data to gmina…
    2011 powiat disaggregation: 2477 synthetic gmina tables
  Layer 1: building seeds (log-linear interpolation)…
    Seeds: 2557 gminas
  Layer 2: voivodeship marginal scaling…
    Scaled 416 voivodeship-year combinations
  Storing results…
  Aggregating to powiat and voivodeship levels…
    Aggregated: 10226 powiat-years, 432 voiv-years
  Summary: 69039 cell-years for 2557 gminas
  ✓  E_educ_2000 complete

Completed in 3.4s


In [16]:
# ── Validation: E_educ_2000 ──
e_sid = 'E_educ_2000'
source_sid = 'M_educ_2000'

# 1. Coverage
n_gminas = sum(
    1 for tid, rec in db._records.items()
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
    and e_sid in rec.cross_tables and rec.cross_tables[e_sid].years_with_data
)
print(f"{e_sid} coverage: {n_gminas} gminas")

# 2. Provenance
prov_df = est.get_provenance_summary(e_sid)
if not prov_df.empty:
    print(f"\nProvenance (sample years):")
    print(prov_df.head(10).to_string())

# 3. Voivodeship consistency — M_educ_2000 has NO ogółem, so compare sum of all categories
voiv_tids = sorted(tid for tid in db._by_level.get(LEVEL_VOIVODESHIP, set()))
errors = []
for vtid in voiv_tids:
    vrec = db._records.get(vtid)
    if vrec is None:
        continue
    obs_ct = vrec.cross_tables.get(source_sid)
    est_ct = vrec.cross_tables.get(e_sid)
    if obs_ct is None or est_ct is None:
        continue
    for year in PREDICTION_2000_RANGE:
        obs_tbl = obs_ct.tables.get(year)
        est_tbl = est_ct.tables.get(year)
        if obs_tbl is None or np.all(np.isnan(obs_tbl)):
            continue
        if est_tbl is None or np.all(np.isnan(est_tbl)):
            continue
        obs_total = np.nansum(obs_tbl)
        est_total = np.nansum(est_tbl)
        if obs_total > 0:
            pct_err = 100 * (est_total - obs_total) / obs_total
            errors.append({'voiv': vtid, 'name': vrec.name, 'year': year,
                           'obs': obs_total, 'est': est_total, 'pct_err': pct_err})

if errors:
    err_df = pd.DataFrame(errors)
    print(f"\nVoivodeship consistency: {len(err_df)} comparisons")
    print(f"  Mean abs % error: {err_df['pct_err'].abs().mean():.4f}%")
    print(f"  Max  abs % error: {err_df['pct_err'].abs().max():.4f}%")
    worst = err_df.loc[err_df['pct_err'].abs().idxmax()]
    print(f"  Worst: {worst['name']} yr={worst['year']}: {worst['pct_err']:.3f}%")

# 4. Spot-check: per-category voivodeship breakdown for a census year
sample_year = 2002
print(f"\nPer-category voivodeship check (year={sample_year}):")
for vtid in voiv_tids[:4]:
    vrec = db._records[vtid]
    obs_ct = vrec.cross_tables.get(source_sid)
    est_ct = vrec.cross_tables.get(e_sid)
    if obs_ct is None or est_ct is None:
        continue
    obs_tbl = obs_ct.tables.get(sample_year)
    est_tbl = est_ct.tables.get(sample_year)
    if obs_tbl is None or est_tbl is None:
        continue
    print(f"  {vrec.name}: obs={np.array2string(obs_tbl, precision=0, separator=', ')}")
    print(f"  {' '*len(vrec.name)}: est={np.array2string(est_tbl, precision=0, separator=', ')}")

E_educ_2000 coverage: 2557 gminas

Provenance (sample years):
      n_units  mean_frac_observed  min_frac_observed
year                                                
1999     2954            0.000000                0.0
2000     2954            0.000000                0.0
2001     2954            0.000000                0.0
2002     2954            0.838863                0.0
2003     2954            0.000000                0.0
2004     2954            0.000000                0.0
2005     2954            0.000000                0.0
2006     2954            0.000000                0.0
2007     2954            0.000000                0.0
2008     2954            0.000000                0.0

Voivodeship consistency: 416 comparisons
  Mean abs % error: 0.0000%
  Max  abs % error: 0.0000%
  Worst: MAZOWIECKIE yr=2021: -0.000%

Per-category voivodeship check (year=2002):
  DOLNOŚLĄSKIE: obs=[698000., 539000., 214000., 651000., 177000.]
              : est=[698000., 539000., 214000., 651000.

## Education — Prediction1990 (1986–2002)

Source: M_educ_1990 (1D, 6 labels with ogółem): P2885 (1988) + P2402 (2002) + H_sex_educ (old voivodeship).  
Strategy: Layer 1 log-linear interpolation → Layer 2 national marginal scaling via M_educ_1990 at country level.  
Consistency check: national-level totals and ogółem vs population.

In [17]:
# ── Run Education Prediction1990 ──
t0 = time.time()
est.run_pipeline('educ', '1990')
print(f"\nCompleted in {time.time()-t0:.1f}s")


  PIPELINE: educ / Prediction1990
  Output subject: E_educ_1990
  Source: M_educ_1990  shape=(6,)
  Gminas total: 2671, with M_educ_1990: 2531
  Layer 1: generating seeds (log-linear interpolation)…
    Seeds generated: 2531/2531 units (skipped 0)
  Layer 2: national marginal scaling…
    Scaled 7 national-year combinations
  Storing results…
  Aggregating to powiat and voivodeship levels…
    Aggregated: 6348 powiat-years, 272 voiv-years
  Summary: 43027 cell-years for 2531 gminas
  ✓  E_educ_1990 complete

Completed in 1.3s


In [18]:
# ── Validation: E_educ_1990 ──
e_sid = 'E_educ_1990'
source_sid = 'M_educ_1990'

# 1. Coverage
n_gminas = sum(
    1 for tid, rec in db._records.items()
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
    and e_sid in rec.cross_tables and rec.cross_tables[e_sid].years_with_data
)
print(f"{e_sid} coverage: {n_gminas} gminas")

# 2. Provenance
prov_df = est.get_provenance_summary(e_sid)
if not prov_df.empty:
    print(f"\nProvenance (sample years):")
    print(prov_df.head(10).to_string())

# 3. National consistency — M_educ_1990 has ogółem at country level (0)
country_rec = db._records.get('0000000')
obs_ct = country_rec.cross_tables.get(source_sid) if country_rec else None
if obs_ct:
    labels = obs_ct.dim_labels.get(obs_ct.dim_names[0], [])
    og_idx = next((i for i, l in enumerate(labels) if l.lower() == 'ogółem'), None)

    for yr in [1988, 1995, 2000, 2002]:
        obs_tbl = obs_ct.tables.get(yr)
        if obs_tbl is None or np.all(np.isnan(obs_tbl)):
            continue
        # Aggregate all gmina E_ tables
        agg = np.zeros_like(obs_tbl)
        n_agg = 0
        for tid, rec in db._records.items():
            if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
                continue
            ect = rec.cross_tables.get(e_sid)
            if ect is None:
                continue
            etbl = ect.tables.get(yr)
            if etbl is None or np.all(np.isnan(etbl)):
                continue
            agg += np.nan_to_num(etbl, nan=0.0)
            n_agg += 1
        
        obs_total = obs_tbl[og_idx] if og_idx is not None else np.nansum(obs_tbl)
        agg_total = agg[og_idx] if og_idx is not None else np.nansum(agg)
        pct = 100 * (agg_total - obs_total) / obs_total if obs_total > 0 else 0.0
        print(f"\n  Year {yr}: national obs ogółem={obs_total:,.0f}, Σ gmina E_={agg_total:,.0f}, "
              f"diff={pct:+.3f}%, n_gminas={n_agg}")

# 4. Internal consistency: ogółem = sum(sub-categories) for sample gminas
print(f"\nInternal ogółem consistency (sample):")
n_checked = 0
n_ok = 0
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect = rec.cross_tables.get(e_sid)
    if ect is None:
        continue
    labels = ect.dim_labels.get(ect.dim_names[0], [])
    ogi = next((i for i, l in enumerate(labels) if l.lower() == 'ogółem'), None)
    if ogi is None:
        continue
    for yr in ect.years_with_data:
        tbl = ect.tables.get(yr)
        if tbl is None:
            continue
        og_val = tbl[ogi]
        sub_sum = np.nansum([tbl[i] for i in range(len(tbl)) if i != ogi])
        n_checked += 1
        if abs(og_val - sub_sum) < 1.0:
            n_ok += 1
print(f"  {n_ok}/{n_checked} within 1.0 tolerance")

E_educ_1990 coverage: 2531 gminas

Provenance (sample years):
      n_units  mean_frac_observed  min_frac_observed
year                                                
1986     2928             0.00000                0.0
1987     2928             0.00000                0.0
1988     2928             0.85041                0.0
1989     2928             0.00000                0.0
1990     2928             0.00000                0.0
1991     2928             0.00000                0.0
1992     2928             0.00000                0.0
1993     2928             0.00000                0.0
1994     2928             0.00000                0.0
1995     2928             0.00000                0.0

  Year 1988: national obs ogółem=28,166,213, Σ gmina E_=28,166,213, diff=+0.000%, n_gminas=2531

Internal ogółem consistency (sample):
  27841/43027 within 1.0 tolerance


## Education × Sex — Prediction2000 (1999–2025)

Source: M_educ_sex_2000 (2D, 5 educ × 3 sex): P2402 + P3309 + P4315 (sex-disaggregated).  
Strategy: 2011 powiat census disaggregated to gmina → spline interpolation → Layer 2 education-marginal scaling via M_educ_2000.  
Consistency check: education marginals (sum over sex=ogółem) should align with E_educ_2000.

In [19]:
# ── Run Education × Sex Prediction2000 ──
t0 = time.time()
est.run_pipeline('educ_sex', '2000')
print(f"\nCompleted in {time.time()-t0:.1f}s")


  PIPELINE: educ_sex / Prediction2000
  Output subject: E_educ_sex_2000
  Source: M_educ_sex_2000  shape=(5, 3)
  Disaggregating 2011 powiat data to gmina…
    2011 powiat disaggregation: 2477 synthetic gmina tables
  Layer 1: building seeds (log-linear interpolation)…
    Seeds: 2557 gminas
  Layer 2: voivodeship education-marginal scaling…
    Scaled 416 voivodeship-year combinations
  Storing results…
  Aggregating to powiat and voivodeship levels…
    Aggregated: 10226 powiat-years, 432 voiv-years
  Summary: 69039 cell-years for 2557 gminas
  ✓  E_educ_sex_2000 complete

Completed in 5.8s


In [20]:
# ── Validation: E_educ_sex_2000 ──
e_sid = 'E_educ_sex_2000'
source_sid = 'M_educ_sex_2000'
educ_sid = 'E_educ_2000'  # for cross-consistency

# 1. Coverage
n_gminas = sum(
    1 for tid, rec in db._records.items()
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
    and e_sid in rec.cross_tables and rec.cross_tables[e_sid].years_with_data
)
print(f"{e_sid} coverage: {n_gminas} gminas")

# 2. Provenance
prov_df = est.get_provenance_summary(e_sid)
if not prov_df.empty:
    print(f"\nProvenance (sample years):")
    print(prov_df.head(10).to_string())

# 3. Voivodeship consistency — compare sex=ogółem slice with E_educ_2000
voiv_tids = sorted(tid for tid in db._by_level.get(LEVEL_VOIVODESHIP, set()))
errors = []
for vtid in voiv_tids:
    vrec = db._records.get(vtid)
    if vrec is None:
        continue
    est_ct = vrec.cross_tables.get(e_sid)
    educ_ct = vrec.cross_tables.get(educ_sid)
    if est_ct is None or educ_ct is None:
        continue
    
    # Find ogółem sex index in 2D table
    dim_names = est_ct.dim_names
    sex_dim = dim_names[1]  # n2 = sex
    sex_labels = est_ct.dim_labels[sex_dim]
    og_sex_idx = next((i for i, l in enumerate(sex_labels) if l.lower() == 'ogółem'), None)
    
    for year in [2002, 2011, 2021]:
        est_tbl = est_ct.tables.get(year)
        educ_tbl = educ_ct.tables.get(year)
        if est_tbl is None or educ_tbl is None:
            continue
        if np.all(np.isnan(est_tbl)) or np.all(np.isnan(educ_tbl)):
            continue
        # Extract sex=ogółem column from 2D E_educ_sex_2000
        if og_sex_idx is not None:
            educ_from_2d = est_tbl[:, og_sex_idx]
        else:
            educ_from_2d = est_tbl.sum(axis=1)  # fallback: sum over sex
        
        educ_from_1d = educ_tbl  # E_educ_2000 is 1D
        if educ_from_2d.shape != educ_from_1d.shape:
            continue
        max_diff = np.abs(educ_from_2d - educ_from_1d).max()
        total = np.nansum(educ_from_1d)
        pct = 100 * max_diff / total if total > 0 else 0
        errors.append({'voiv': vtid, 'year': year, 'max_cell_diff': max_diff, 'pct_of_total': pct})

if errors:
    err_df = pd.DataFrame(errors)
    print(f"\nEduc marginal consistency (E_educ_sex vs E_educ): {len(err_df)} comparisons")
    print(f"  Mean max cell diff: {err_df['max_cell_diff'].mean():.1f}")
    print(f"  Max max cell diff:  {err_df['max_cell_diff'].max():.1f}")
    print(f"  Mean % of total:    {err_df['pct_of_total'].mean():.4f}%")

# 4. Shape check
sample_rec = None
for tid, rec in db._records.items():
    if rec.level == LEVEL_GMINA and e_sid in rec.cross_tables:
        sample_rec = rec
        break
if sample_rec:
    ct = sample_rec.cross_tables[e_sid]
    print(f"\nShape: {ct._shape}, dim_names: {ct.dim_names}")
    for d in ct.dim_names:
        print(f"  {d}: {ct.dim_labels[d]}")

E_educ_sex_2000 coverage: 2557 gminas

Provenance (sample years):
      n_units  mean_frac_observed  min_frac_observed
year                                                
1999     2954            0.000000                0.0
2000     2954            0.000000                0.0
2001     2954            0.000000                0.0
2002     2954            0.838863                0.0
2003     2954            0.000000                0.0
2004     2954            0.000000                0.0
2005     2954            0.000000                0.0
2006     2954            0.000000                0.0
2007     2954            0.000000                0.0
2008     2954            0.000000                0.0

Educ marginal consistency (E_educ_sex vs E_educ): 48 comparisons
  Mean max cell diff: 0.0
  Max max cell diff:  0.0
  Mean % of total:    0.0000%

Shape: (5, 3), dim_names: ['n1', 'n2']
  n1: ['gimnazjalne, podstawowe i niższe', 'policealne oraz średnie zawodowe/branżowe', 'wyższe', 'zasadnicze 

## Education × Sex — Prediction1990 (1986–2002)

Source: M_educ_sex_1990 (2D, 6 educ × 3 sex): P2402 (2002, all sex) + H_sex_educ (old voivodeship).  
Strategy: Phase A 1988 IPF (national M_educ_sex_1990 seed + M_educ_1990 gmina marginals) → Phase B interpolation → Phase C national scaling via M_educ_sex_1990.  
Most complex: requires IPF seeding from national marginals down to gmina level.

In [21]:
# ── Run Education × Sex Prediction1990 ──
t0 = time.time()
est.run_pipeline('educ_sex', '1990')
print(f"\nCompleted in {time.time()-t0:.1f}s")


  PIPELINE: educ_sex / Prediction1990
  Output subject: E_educ_sex_1990
  Source: M_educ_sex_1990  shape=(6, 3)
  Phase A: constructing 1988 gmina educ×sex via IPF…
    1988 IPF: 2489 OK, 182 skipped
  Phase B: building seeds (log-linear interpolation)…
    Seeds: 2531 gminas
  Phase C: national marginal scaling…
    Scaled 7 national-year combinations
  Phase D: storing results…
  Aggregating to powiat and voivodeship levels…
    Aggregated: 6348 powiat-years, 272 voiv-years
  Summary: 43027 cell-years for 2531 gminas
  ✓  E_educ_sex_1990 complete

Completed in 1.6s


In [22]:
# ── Validation: E_educ_sex_1990 ──
e_sid = 'E_educ_sex_1990'
source_sid = 'M_educ_sex_1990'
educ_1d_sid = 'E_educ_1990'

# 1. Coverage
n_gminas = sum(
    1 for tid, rec in db._records.items()
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
    and e_sid in rec.cross_tables and rec.cross_tables[e_sid].years_with_data
)
print(f"{e_sid} coverage: {n_gminas} gminas")

# 2. Provenance
prov_df = est.get_provenance_summary(e_sid)
if not prov_df.empty:
    print(f"\nProvenance (sample years):")
    print(prov_df.head(10).to_string())

# 3. National consistency — compare Σ gmina with country-level M_educ_sex_1990
country_rec = db._records.get('0000000')
obs_ct = country_rec.cross_tables.get(source_sid) if country_rec else None
if obs_ct:
    for yr in [1988, 1995, 2002]:
        obs_tbl = obs_ct.tables.get(yr)
        if obs_tbl is None or np.all(np.isnan(obs_tbl)):
            continue
        agg = np.zeros_like(obs_tbl)
        n_agg = 0
        for tid, rec in db._records.items():
            if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
                continue
            ect = rec.cross_tables.get(e_sid)
            if ect is None:
                continue
            etbl = ect.tables.get(yr)
            if etbl is None or np.all(np.isnan(etbl)) or etbl.shape != obs_tbl.shape:
                continue
            agg += np.nan_to_num(etbl, nan=0.0)
            n_agg += 1
        
        obs_total = np.nansum(obs_tbl)
        agg_total = np.nansum(agg)
        pct = 100 * (agg_total - obs_total) / obs_total if obs_total > 0 else 0
        print(f"\n  Year {yr}: national obs total={obs_total:,.0f}, Σ gmina={agg_total:,.0f}, "
              f"diff={pct:+.3f}%, n_gminas={n_agg}")

# 4. Cross-consistency with E_educ_1990: educ marginal (sum over sex=ogółem) should match
print(f"\nEduc marginal cross-check with {educ_1d_sid}:")
n_match = 0
n_total = 0
max_diff = 0.0
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect_2d = rec.cross_tables.get(e_sid)
    ect_1d = rec.cross_tables.get(educ_1d_sid)
    if ect_2d is None or ect_1d is None:
        continue
    dim_names = ect_2d.dim_names
    sex_labels = ect_2d.dim_labels.get(dim_names[1], [])
    og_sex = next((i for i, l in enumerate(sex_labels) if l.lower() == 'ogółem'), None)
    for yr in ect_2d.years_with_data:
        if yr not in ect_1d.years_with_data:
            continue
        tbl_2d = ect_2d.tables.get(yr)
        tbl_1d = ect_1d.tables.get(yr)
        if tbl_2d is None or tbl_1d is None:
            continue
        educ_from_2d = tbl_2d[:, og_sex] if og_sex is not None else tbl_2d.sum(axis=1)
        if educ_from_2d.shape == tbl_1d.shape:
            diff = np.abs(educ_from_2d - tbl_1d).max()
            n_total += 1
            if diff < 1.0:
                n_match += 1
            max_diff = max(max_diff, diff)
    
print(f"  {n_match}/{n_total} match within 1.0, max cell diff={max_diff:.2f}")

E_educ_sex_1990 coverage: 2531 gminas

Provenance (sample years):
      n_units  mean_frac_observed  min_frac_observed
year                                                
1986     2928                 0.0                0.0
1987     2928                 0.0                0.0
1988     2928                 0.0                0.0
1989     2928                 0.0                0.0
1990     2928                 0.0                0.0
1991     2928                 0.0                0.0
1992     2928                 0.0                0.0
1993     2928                 0.0                0.0
1994     2928                 0.0                0.0
1995     2928                 0.0                0.0

  Year 1988: national obs total=112,869,976, Σ gmina=112,664,852, diff=-0.182%, n_gminas=2531

Educ marginal cross-check with E_educ_1990:
  8247/43027 match within 1.0, max cell diff=541.40


## Household Size — Prediction2000 (1999–2025)

Source: M_hh_size_2000 (1D, 6 categories with ogółem): P2871 + P3420 + P4287.  
Strategy: 2011 powiat census disaggregated to gmina → 3-anchor spline interpolation. **No marginal scaling** (no higher-level marginals).  
Note: ogółem = total **households** (not population). No population scaling applied.

In [23]:
# ── Run Household Size Prediction2000 ──
t0 = time.time()
est.run_pipeline('hh_size', '2000')
print(f"\nCompleted in {time.time()-t0:.1f}s")


  PIPELINE: hh_size / Prediction2000
  Output subject: E_hh_size_2000
  Source: M_hh_size_2000  shape=(6,)
  Disaggregating 2011 powiat data to gmina…
    2011 powiat disaggregation: 2477 synthetic gmina tables
  Layer 1: building seeds (log-linear interpolation)…
    Seeds: 2557 gminas
  Storing results…
  Aggregating to powiat and voivodeship levels…
    Aggregated: 10226 powiat-years, 432 voiv-years
  Summary: 69039 cell-years for 2557 gminas
  ✓  E_hh_size_2000 complete

Completed in 3.4s


In [24]:
# ── Validation: E_hh_size_2000 ──
e_sid = 'E_hh_size_2000'
source_sid = 'M_hh_size_2000'

# 1. Coverage
n_gminas = sum(
    1 for tid, rec in db._records.items()
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
    and e_sid in rec.cross_tables and rec.cross_tables[e_sid].years_with_data
)
print(f"{e_sid} coverage: {n_gminas} gminas")

# 2. Provenance
prov_df = est.get_provenance_summary(e_sid)
if not prov_df.empty:
    print(f"\nProvenance (sample years):")
    print(prov_df.head(10).to_string())

# 3. Internal consistency: ogółem = sum(sub-categories)
print(f"\nInternal ogółem consistency:")
n_checked = 0
n_ok = 0
max_err = 0.0
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect = rec.cross_tables.get(e_sid)
    if ect is None:
        continue
    labels = ect.dim_labels.get(ect.dim_names[0], [])
    ogi = next((i for i, l in enumerate(labels) if l.lower() == 'ogółem'), None)
    if ogi is None:
        continue
    for yr in ect.years_with_data:
        tbl = ect.tables.get(yr)
        if tbl is None:
            continue
        og_val = tbl[ogi]
        sub_sum = np.nansum([tbl[i] for i in range(len(tbl)) if i != ogi])
        n_checked += 1
        err = abs(og_val - sub_sum)
        max_err = max(max_err, err)
        if err < 1.0:
            n_ok += 1
print(f"  {n_ok}/{n_checked} within 1.0 tolerance, max error={max_err:.2f}")

# 4. Anchor year check: E_ should match M_ at observed years
print(f"\nAnchor year match (census years):")
for census_yr in [2002, 2011, 2021]:
    n_match = 0
    n_total = 0
    worst_diff = 0.0
    for tid, rec in db._records.items():
        if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
            continue
        ect = rec.cross_tables.get(e_sid)
        oct = rec.cross_tables.get(source_sid)
        if ect is None or oct is None:
            continue
        etbl = ect.tables.get(census_yr)
        otbl = oct.tables.get(census_yr)
        if etbl is None or otbl is None:
            continue
        if np.all(np.isnan(etbl)) or np.all(np.isnan(otbl)):
            continue
        n_total += 1
        diff = np.nanmax(np.abs(etbl - otbl))
        worst_diff = max(worst_diff, diff)
        if diff < 1.0:
            n_match += 1
    if n_total > 0:
        print(f"  {census_yr}: {n_match}/{n_total} match (<1.0), worst cell diff={worst_diff:.2f}")

E_hh_size_2000 coverage: 2557 gminas

Provenance (sample years):
      n_units  mean_frac_observed  min_frac_observed
year                                                
1999     2954            0.000000                0.0
2000     2954            0.000000                0.0
2001     2954            0.000000                0.0
2002     2954            0.838863                0.0
2003     2954            0.000000                0.0
2004     2954            0.000000                0.0
2005     2954            0.000000                0.0
2006     2954            0.000000                0.0
2007     2954            0.000000                0.0
2008     2954            0.000000                0.0

Internal ogółem consistency:
  69039/69039 within 1.0 tolerance, max error=0.00

Anchor year match (census years):
  2002: 2478/2478 match (<1.0), worst cell diff=0.00
  2021: 2477/2477 match (<1.0), worst cell diff=0.00


## Household Size — Prediction1990 (1986–2002)

Source: M_hh_size_1990 (1D, 5 categories with ogółem): P2887 (1988) + P2871 (2002).  
Strategy: Pure Layer 1 log-linear interpolation between 1988 and 2002 anchors. **No marginals, no population scaling.**  
Simplest pipeline: only two anchor points are available per gmina.

In [25]:
# ── Run Household Size Prediction1990 ──
t0 = time.time()
est.run_pipeline('hh_size', '1990')
print(f"\nCompleted in {time.time()-t0:.1f}s")


  PIPELINE: hh_size / Prediction1990
  Output subject: E_hh_size_1990
  Source: M_hh_size_1990  shape=(5,)
  Gminas total: 2671, with M_hh_size_1990: 2531
  Layer 1: generating seeds (log-linear interpolation)…
    Seeds generated: 2531/2531 units (skipped 0)
  Storing results…
  Aggregating to powiat and voivodeship levels…
    Aggregated: 6348 powiat-years, 272 voiv-years
  Summary: 43027 cell-years for 2531 gminas
  ✓  E_hh_size_1990 complete

Completed in 1.4s


In [26]:
# ── Validation: E_hh_size_1990 ──
e_sid = 'E_hh_size_1990'
source_sid = 'M_hh_size_1990'

# 1. Coverage
n_gminas = sum(
    1 for tid, rec in db._records.items()
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
    and e_sid in rec.cross_tables and rec.cross_tables[e_sid].years_with_data
)
print(f"{e_sid} coverage: {n_gminas} gminas")

# 2. Provenance
prov_df = est.get_provenance_summary(e_sid)
if not prov_df.empty:
    print(f"\nProvenance (sample years):")
    print(prov_df.head(10).to_string())

# 3. Internal consistency: ogółem = sum(sub-categories)
print(f"\nInternal ogółem consistency:")
n_checked = 0
n_ok = 0
max_err = 0.0
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect = rec.cross_tables.get(e_sid)
    if ect is None:
        continue
    labels = ect.dim_labels.get(ect.dim_names[0], [])
    ogi = next((i for i, l in enumerate(labels) if l.lower() == 'ogółem'), None)
    if ogi is None:
        continue
    for yr in ect.years_with_data:
        tbl = ect.tables.get(yr)
        if tbl is None:
            continue
        og_val = tbl[ogi]
        sub_sum = np.nansum([tbl[i] for i in range(len(tbl)) if i != ogi])
        n_checked += 1
        err = abs(og_val - sub_sum)
        max_err = max(max_err, err)
        if err < 1.0:
            n_ok += 1
print(f"  {n_ok}/{n_checked} within 1.0 tolerance, max error={max_err:.2f}")

# 4. Anchor year exact match (1988 and 2002)
print(f"\nAnchor year match:")
for census_yr in [1988, 2002]:
    n_match = 0
    n_total = 0
    worst_diff = 0.0
    for tid, rec in db._records.items():
        if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
            continue
        ect = rec.cross_tables.get(e_sid)
        oct = rec.cross_tables.get(source_sid)
        if ect is None or oct is None:
            continue
        etbl = ect.tables.get(census_yr)
        otbl = oct.tables.get(census_yr)
        if etbl is None or otbl is None:
            continue
        if np.all(np.isnan(etbl)) or np.all(np.isnan(otbl)):
            continue
        n_total += 1
        diff = np.nanmax(np.abs(etbl - otbl))
        worst_diff = max(worst_diff, diff)
        if diff < 1.0:
            n_match += 1
    if n_total > 0:
        print(f"  {census_yr}: {n_match}/{n_total} match (<1.0), worst cell diff={worst_diff:.2f}")

# 5. Time series plausibility: show one gmina's trajectory
print(f"\nSample time series:")
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect = rec.cross_tables.get(e_sid)
    if ect is None or len(ect.years_with_data) < 10:
        continue
    labels = ect.dim_labels.get(ect.dim_names[0], [])
    ogi = next((i for i, l in enumerate(labels) if l.lower() == 'ogółem'), None)
    print(f"  {rec.name} ({tid}):")
    for yr in sorted(ect.years_with_data)[:5]:
        tbl = ect.tables.get(yr)
        if tbl is not None:
            og_v = tbl[ogi] if ogi is not None else np.nansum(tbl)
            print(f"    {yr}: ogółem={og_v:,.0f}")
    print(f"    ...")
    for yr in sorted(ect.years_with_data)[-3:]:
        tbl = ect.tables.get(yr)
        if tbl is not None:
            og_v = tbl[ogi] if ogi is not None else np.nansum(tbl)
            print(f"    {yr}: ogółem={og_v:,.0f}")
    break

E_hh_size_1990 coverage: 2531 gminas

Provenance (sample years):
      n_units  mean_frac_observed  min_frac_observed
year                                                
1986     2928             0.00000                0.0
1987     2928             0.00000                0.0
1988     2928             0.85041                0.0
1989     2928             0.00000                0.0
1990     2928             0.00000                0.0
1991     2928             0.00000                0.0
1992     2928             0.00000                0.0
1993     2928             0.00000                0.0
1994     2928             0.00000                0.0
1995     2928             0.00000                0.0

Internal ogółem consistency:
  43027/43027 within 1.0 tolerance, max error=0.00

Anchor year match:
  1988: 2490/2490 match (<1.0), worst cell diff=0.00
  2002: 2478/2478 match (<1.0), worst cell diff=0.00

Sample time series:
  Bolesławiec (0201011):
    1986: ogółem=14,291
    1987: ogółem=14,29

In [27]:
# ── Final Summary: All Estimation Pipelines ──
print("=" * 70)
print("  GUS04D — Variable-Specific Estimation Pipelines — SUMMARY")
print("=" * 70)

all_e_sids = [
    'E_age_sex_2000', 'E_age_sex_1990',
    'E_educ_2000', 'E_educ_1990',
    'E_educ_sex_2000', 'E_educ_sex_1990',
    'E_hh_size_2000', 'E_hh_size_1990',
]

for e_sid in all_e_sids:
    n_gminas = sum(
        1 for tid, rec in db._records.items()
        if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
        and e_sid in rec.cross_tables and rec.cross_tables[e_sid].years_with_data
    )
    # Get shape from first record that has it
    shape_str = '?'
    for tid, rec in db._records.items():
        if e_sid in rec.cross_tables and rec.cross_tables[e_sid].years_with_data:
            ct = rec.cross_tables[e_sid]
            shape_str = str(ct._shape)
            break
    
    completed = e_sid.replace('E_', '').rsplit('_', 1)
    key = (completed[0], completed[1])
    status = "✓" if key in est._completed else "✗"
    print(f"  {status} {e_sid:25s}  shape={shape_str:15s}  gminas={n_gminas}")

print(f"\nPipelines completed: {len(est._completed)}/{len(all_e_sids)}")
print(f"Pipelines: {sorted(est._completed)}")

  GUS04D — Variable-Specific Estimation Pipelines — SUMMARY
  ✓ E_age_sex_2000             shape=(16, 3)          gminas=2671
  ✓ E_age_sex_1990             shape=(16, 3)          gminas=2671
  ✓ E_educ_2000                shape=(5,)             gminas=2557
  ✓ E_educ_1990                shape=(6,)             gminas=2531
  ✓ E_educ_sex_2000            shape=(5, 3)           gminas=2557
  ✓ E_educ_sex_1990            shape=(6, 3)           gminas=2531
  ✓ E_hh_size_2000             shape=(6,)             gminas=2557
  ✓ E_hh_size_1990             shape=(5,)             gminas=2531

Pipelines completed: 8/8
Pipelines: [('age_sex', '1990'), ('age_sex', '2000'), ('educ', '1990'), ('educ', '2000'), ('educ_sex', '1990'), ('educ_sex', '2000'), ('hh_size', '1990'), ('hh_size', '2000')]


---

## Summary

| Pipeline | E_ Subject | Shape | Layers | Key Features |
|---|---|---|---|---|
| Age×Sex 2000 | E_age_sex_2000 | (19, 3) | L1 + L3 | Light residual scaling, voivodeship hierarchy |
| Age×Sex 1990 | E_age_sex_1990 | (19, 3) | L1 + L2 + L3 | Phase A IPF (1988 census) → interpolation → old voivodeship scaling |
| Education 2000 | E_educ_2000 | (5,) | L1 + L2 | 2011 disaggregation + voivodeship marginal scaling |
| Education 1990 | E_educ_1990 | (6,) | L1 + L2 | Interpolation + national marginal scaling |
| Education×Sex 2000 | E_educ_sex_2000 | (5, 3) | L1 + L2 | 2011 disaggregation + education-marginal scaling via M_educ_2000 |
| Education×Sex 1990 | E_educ_sex_1990 | (6, 3) | L1 + L2 | Phase A IPF (national→gmina) → interpolation → national scaling |
| HH Size 2000 | E_hh_size_2000 | (6,) | L1 | 2011 disaggregation + spline interpolation (no marginals) |
| HH Size 1990 | E_hh_size_1990 | (5,) | L1 | Pure interpolation (1988→2002, no marginals) |

**Note:** `E_age_educ_2000` is deferred — requires M_pop__age_educ data not yet in the database.